In [1]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
from itertools import groupby
from log_kde import log_kde
from scipy.special import kl_div
from tqdm.notebook import tqdm

In [2]:
MarkovChain = "P_matrix_cyclooctane_R144_MSM2.csv"
P_ref = np.loadtxt(f"MarkovChains/{MarkovChain}", delimiter=",")
P_ref = P_ref.astype("float64")

P_ref.shape

(2501, 2501)

$$\log_{kde}(p) = \psi(p) = \log\left(\sum_{i=1}^N c_i \exp(-KL(p \;||\; P_{i, -}))\right)$$

In [3]:
# log kde object to generate R, Q
time = 500
log_kde_obj = log_kde(P_ref, time)

In [ ]:
Q_mat = log_kde_obj.calculate_Q_mat()

In [5]:
def KL_map(P, Q):
    indices = []
    min_costs = []
    for p_i in tqdm(P):
        costs = kl_div(Q, p_i).sum(axis=1)
        indices.append(costs.argmin())
        min_costs.append(costs.min())
    return indices, min_costs

In [6]:
def parse_dump(dump_file):
    timesteps = []

    # read through dump_file
    with open(dump_file, 'r') as file:
        trajectory = file.readlines()

        current_timestep = ""
        counter = 0
        for line in trajectory:
            if line == "ITEM: TIMESTEP\n" and current_timestep:
                timesteps.append(current_timestep)
                counter += 1
                current_timestep = ""
            current_timestep += line

        timesteps.append(current_timestep)
    return timesteps

In [7]:
# takes NDarray
# needs a set of points
def gram_schmidt_orthonormalization(P):
    # Step 1: translate
    P_shifted = P - P[0]

    # Step 2: construct orthonormal frame
    p2 = P_shifted[1]
    p3 = P_shifted[2]
    z = np.cross(p2, p3)

    x_hat = p2 / np.linalg.norm(p2)
    z_hat = z / np.linalg.norm(z)
    y_hat = np.cross(z_hat, x_hat)

    # Rotation matrix
    R_mat = np.vstack([x_hat, y_hat, z_hat])

    # Apply rotation
    P_final = P_shifted @ R_mat.T

    return P_final.reshape(1, -1)

In [8]:
# get ordering for movie via KL cost
indices_Q_mat, cost_Q_mat = KL_map(log_kde_obj.Pt, Q_mat)

  0%|          | 0/2501 [00:00<?, ?it/s]

In [9]:
trajectories = parse_dump('trajectories/pos_cyclooctane.dump')
# clean out duplicates using groupby
trajectories_reordered = [trajectories[i] for i, _ in groupby(indices_Q_mat)]

In [10]:
with open("pos_cyclooctane_reordered_KL_norm.dump", "w") as file:
    file.writelines(trajectories_reordered)

In [11]:
with open("alpha_movie.dump", "w") as alpha_movie:
    for traj in parse_dump("pos_cyclooctane_reordered.dump"):
        # step information
        timestep = 0
        num_atoms = -1
        pos_atoms = []
        type_atoms = []
        grab_pos = False

        traj = traj.split("\n")
        for i, line in enumerate(traj):
            # grabs atom type and xyz
            if grab_pos and line:
                type_atoms.append(line.split(" ")[-4])
                pos_atoms.append(line.split(" ")[-3:])

            # grabs timestep
            elif line == "ITEM: TIMESTEP":
                timestep = traj[i + 1]

            # grabs number of atoms
            elif line == "ITEM: NUMBER OF ATOMS":
                num_atoms = traj[i + 1]

            # enables position grabbing, positions are always last lines in a timestep
            elif "ITEM: ATOMS" in line:
                grab_pos = True
            
        positions = np.array(pos_atoms, dtype=np.float64)
        shifted_positions = gram_schmidt_orthonormalization(positions).reshape(-1, 3)
        # write to file
        alpha_movie.write(f"{num_atoms}\n")
        alpha_movie.write(f"timestep {timestep}\n")
        for type, pos in zip(type_atoms, shifted_positions):
            alpha_movie.write(f"{type} {" ".join(map(str, pos))}\n")

In [12]:
shifted_positions = gram_schmidt_orthonormalization(positions).reshape(-1, 3)

In [13]:
# extract new basis vectors for each frame
# take the average(0, 1) to average(4, 5) = a
# take the average(2, 3) to average(6, 7) = b
def generate_basis_vectors(carbons):
    # x basis vector
    midpoint_p0p1 = (carbons[0] + carbons[1]) / 2
    midpoint_p4p5 = (carbons[4] + carbons[5]) / 2
    a = midpoint_p4p5 - midpoint_p0p1

    # y basis vector
    midpoint_p2p3 = (carbons[2] + carbons[3]) / 2
    midopint_p6p7 = (carbons[6] + carbons[7]) / 2
    b = midopint_p6p7 - midpoint_p2p3

    return a, b

In [14]:
# builds a new basis
def build_rotation_matrix(a, b):
    """Build rotation matrix from two new axis directions (x, y), leave z free."""
    u1 = a / np.linalg.norm(a)

    # Gram-Schmidt: remove u1 component from b
    u2 = b - np.dot(b, u1) * u1
    u2 = u2 / np.linalg.norm(u2)

    # Stack as rows; z row passes through
    R = np.array([
        [u1[0], u1[1], u1[2]],
        [u2[0], u2[1], u2[2]],
        [0,     0,     1    ],
    ])
    return R

In [15]:
# transform the points
def transform_points(points, a, b, origin=None):
    """
    points: (N, 3) array
    a, b:   new x and y axis directions (3D vectors)
    origin: optional point to subtract before rotating
    """
    R = build_rotation_matrix(a, b)
    if origin is not None:
        points = points - origin
    return (R @ points.T).T

In [16]:
with open("alpha_movie_rotated_KL_norm.dump", "w") as alpha_movie:
    for traj in parse_dump("pos_cyclooctane_reordered_KL.dump"):
        # step information
        timestep = 0
        num_atoms = -1
        pos_atoms = []
        type_atoms = []
        grab_pos = False

        traj = traj.split("\n")
        for i, line in enumerate(traj):
            # grabs atom type and xyz
            if grab_pos and line:
                type_atoms.append(line.split(" ")[-4])
                pos_atoms.append(line.split(" ")[-3:])

            # grabs timestep
            elif line == "ITEM: TIMESTEP":
                timestep = traj[i + 1]

            # grabs number of atoms
            elif line == "ITEM: NUMBER OF ATOMS":
                num_atoms = traj[i + 1]

            # enables position grabbing, positions are always last lines in a timestep
            elif "ITEM: ATOMS" in line:
                grab_pos = True

        # rotation operations 
        positions = np.array(pos_atoms, dtype=np.float64)
        carbons = positions[0:8]
        a, b = generate_basis_vectors(carbons)
        rotation_matrix = build_rotation_matrix(a, b)

        rotated_positions = (rotation_matrix @ positions.T).T
        # write to file
        alpha_movie.write(f"{num_atoms}\n")
        alpha_movie.write(f"timestep {timestep}\n")
        for type, pos in zip(type_atoms, rotated_positions):
            alpha_movie.write(f"{type} {" ".join(map(str, pos))}\n")